# Analyse rapide d'une action basée sur 4 indicateurs clés

Ce notebook récupère des informations depuis Yahoo Finance avec `yfinance` et présente : la capitalisation boursière, le PER, le rendement du dividende et la position du cours dans sa fourchette des 52 dernières semaines.

> Les données peuvent être retardées, absentes ou révisées. Cette analyse est indicative et ne constitue pas un conseil financier. Le PER et le rendement du dividende doivent être interprétés selon le secteur et la situation de l'entreprise.

## 1. Installer la dépendance (si nécessaire)

Exécute cette cellule une seule fois si `yfinance` n'est pas déjà installé dans ton environnement.

In [ ]:
%pip install -q yfinance pandas

## 2. Choisir l'action

Modifie `SYMBOL` ci-dessous. Exemples : `AAPL` (Apple), `MSFT` (Microsoft), `MC.PA` (LVMH à Paris), `AIR.PA` (Airbus à Paris). Le suffixe dépend de la place boursière.

In [ ]:
SYMBOL = "AAPL"  # Remplace par le symbole Yahoo Finance souhaité

import math
import pandas as pd
import yfinance as yf

ticker = yf.Ticker(SYMBOL)
info = ticker.get_info()

if not info:
    raise ValueError(f"Aucune information trouvée pour {SYMBOL!r}. Vérifie le symbole Yahoo Finance.")

nom = info.get("shortName") or info.get("longName") or SYMBOL
devise = info.get("currency", "")
print(f"Société : {nom} ({SYMBOL.upper()})")
print(f"Devise des cours/valeurs : {devise or 'non précisée'}")

## 3. Récupérer et afficher les quatre indicateurs

Le PER affiché en premier est le PER historique (*trailing P/E*). Si Yahoo Finance ne le fournit pas, le notebook utilise le PER prévisionnel (*forward P/E*) et le précise. Pour le dividende, Yahoo Finance renvoie généralement un ratio décimal, converti ici en pourcentage.

In [ ]:
def nombre(valeur):
    """Convertit une valeur en nombre exploitable, sinon renvoie None."""
    try:
        resultat = float(valeur)
        return resultat if math.isfinite(resultat) else None
    except (TypeError, ValueError):
        return None

def format_montant(valeur, devise=devise):
    if valeur is None:
        return "Indisponible"
    return f"{valeur:,.2f} {devise}".strip()

# 1) Capitalisation boursière
capitalisation = nombre(info.get("marketCap"))

# 2) PER : privilégier le PER historique, sinon afficher le PER prévisionnel
per_historique = nombre(info.get("trailingPE"))
per_previsionnel = nombre(info.get("forwardPE"))
if per_historique is not None:
    per = per_historique
    type_per = "historique"
else:
    per = per_previsionnel
    type_per = "prévisionnel (si disponible)"

# 3) Rendement du dividende. Certains résultats Yahoo sont exprimés en décimal.
rendement = nombre(info.get("dividendYield"))
if rendement is None:
    rendement_pct = None
else:
    rendement_pct = rendement * 100 if abs(rendement) <= 1 else rendement

# 4) Cours actuel et bornes 52 semaines (données info, avec historique en secours)
cours = (nombre(info.get("currentPrice")) or
         nombre(info.get("regularMarketPrice")) or
         nombre(info.get("previousClose")))
plus_bas = nombre(info.get("fiftyTwoWeekLow"))
plus_haut = nombre(info.get("fiftyTwoWeekHigh"))

if cours is None or plus_bas is None or plus_haut is None:
    historique = ticker.history(period="1y", auto_adjust=False)
    if not historique.empty:
        if cours is None:
            cours = nombre(historique["Close"].dropna().iloc[-1])
        if plus_bas is None:
            plus_bas = nombre(historique["Low"].min())
        if plus_haut is None:
            plus_haut = nombre(historique["High"].max())

print(f"1. Capitalisation boursière : {format_montant(capitalisation)}")
if per is None:
    print("2. PER : Indisponible (par exemple, bénéfices négatifs ou donnée non fournie)")
else:
    print(f"2. PER {type_per} : {per:.2f}")
if rendement_pct is None:
    print("3. Rendement du dividende : Indisponible ou aucun dividende déclaré")
else:
    print(f"3. Rendement du dividende : {rendement_pct:.2f} %")
if cours is not None and plus_bas is not None and plus_haut is not None:
    print(f"4. Cours actuel : {format_montant(cours)}")
    print(f"   Fourchette 52 semaines : {format_montant(plus_bas)} – {format_montant(plus_haut)}")
else:
    print("4. Fourchette 52 semaines : Indisponible")

## 4. Lecture rapide

Les commentaires suivants sont descriptifs, pas des signaux d'achat ou de vente. Un PER élevé peut refléter des anticipations de croissance, tandis qu'un PER faible peut aussi signaler des difficultés. Un rendement élevé n'est pas nécessairement durable.

In [ ]:
print(f"Synthèse indicative — {nom} ({SYMBOL.upper()})")
print("-" * 55)

if capitalisation is None:
    print("• Capitalisation : donnée indisponible.")
else:
    print(f"• Capitalisation : {format_montant(capitalisation)} ; elle donne un ordre de grandeur de la taille de l'entreprise.")

if per is None:
    print("• PER : indisponible ; il peut notamment ne pas être pertinent si les bénéfices sont négatifs.")
elif per < 0:
    print(f"• PER : {per:.2f} ; valeur négative, à interpréter avec prudence (bénéfices négatifs ou données particulières).")
else:
    print(f"• PER {type_per} : {per:.2f} ; à comparer aux entreprises du même secteur et à la croissance attendue.")

if rendement_pct is None:
    print("• Dividende : aucun rendement disponible ; l'entreprise peut ne pas verser de dividende.")
else:
    print(f"• Rendement : {rendement_pct:.2f} % ; ce rendement passé/indiqué ne garantit pas les versements futurs.")

if cours is not None and plus_bas is not None and plus_haut is not None and plus_haut > plus_bas:
    position = (cours - plus_bas) / (plus_haut - plus_bas) * 100
    position = max(0, min(100, position))
    print(f"• Fourchette 52 semaines : cours à {position:.1f} % de l'intervalle entre le plus bas et le plus haut.")
    print("  Une position proche du haut/bas ne permet pas, à elle seule, de prévoir la direction future du cours.")
elif cours is not None:
    print(f"• Cours : {format_montant(cours)} ; bornes 52 semaines indisponibles ou non valides.")
else:
    print("• Fourchette 52 semaines : données insuffisantes pour calculer la position du cours.")

print("\nNote : vérifie les chiffres sur les sources officielles de l'entreprise avant toute décision.")

## Interprétation et limites

- La **capitalisation** est le cours multiplié par le nombre d'actions ; elle varie avec le cours.
- Le **PER** peut être peu significatif pour une entreprise déficitaire et les PER historiques et prévisionnels ne reposent pas sur les mêmes bénéfices.
- Le **rendement du dividende** est une photographie du dividende attendu/indiqué par Yahoo Finance rapporté au cours ; il peut changer ou être supprimé.
- La **fourchette 52 semaines** décrit une période passée, pas un objectif de cours.

Les chiffres sont fournis par Yahoo Finance via `yfinance`, service tiers non affilié à ce notebook. L'accès aux données peut être modifié ou limité par Yahoo Finance.